In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/krrishshandilya/dipcoma-trainval/patch_log.csv
/kaggle/input/datasets/krrishshandilya/dipcoma-trainval/protocol.json
/kaggle/input/datasets/krrishshandilya/dipcoma-trainval/scene_log.csv
/kaggle/input/datasets/krrishshandilya/dipcoma-trainval/code/dipcoma_eval.py
/kaggle/input/datasets/krrishshandilya/dipcoma-trainval/code/dipcoma_learned.py
/kaggle/input/datasets/krrishshandilya/dipcoma-trainval/dataset/train/amazon_manaus/20MRB_20250730_r0_c9.npy
/kaggle/input/datasets/krrishshandilya/dipcoma-trainval/dataset/train/amazon_manaus/20MRB_20250730_r1_c2.npy
/kaggle/input/datasets/krrishshandilya/dipcoma-trainval/dataset/train/amazon_manaus/20MRB_20250730_r1_c3.npy
/kaggle/input/datasets/krrishshandilya/dipcoma-trainval/dataset/train/amazon_manaus/20MRB_20250730_r1_c4.npy
/kaggle/input/datasets/krrishshandilya/dipcoma-trainval/dataset/train/amazon_manaus/20MRB_20250730_r2_c9.npy
/kaggle/input/datasets/krrishshandilya/dipcoma-trainval/dataset/train/amazon_manaus/20MR

In [2]:
import os, sys, glob, time, json, subprocess, importlib.util

# ====== settings ======
TEST = False       # True = quick test (about 10 min, also measures the wide network's speed). Real run: False.
PART = "B"        # real run only: "A" = split lambda 0.01 and 0.04 + second-seed repeats ; "B" = wide merged network
# ======================
STEPS, EVERY, N_EVAL = 15000, 1000, 165
# job = (tag, mode, lambda, steps, N, M, seed)
if TEST:
    JOBS = {0: [("t_wide", "merged", 0.08, 200, 192, 320, 0)], 1: [("t_split", "split", 0.01, 200, 128, 192, 1)]}
    EVERY, N_EVAL, OUT = 100, 4, "/kaggle/working/train_test2"
elif PART == "A":
    JOBS = {0: [("merged_l0.02_s1", "merged", 0.02, STEPS, 128, 192, 1), ("split_l0.01", "split", 0.01, STEPS, 128, 192, 0)],
            1: [("split_l0.04", "split", 0.04, STEPS, 128, 192, 0), ("split_l0.02_s1", "split", 0.02, STEPS, 128, 192, 1)]}
    OUT = "/kaggle/working/trainA"
else:
    JOBS = {0: [("wide_merged_l0.08", "merged", 0.08, STEPS, 192, 320, 0)],
            1: [("wide_merged_l1.28", "merged", 1.28, STEPS, 192, 320, 0)]}
    OUT = "/kaggle/working/trainB"
os.makedirs(OUT, exist_ok=True)
# earlier results for comparison (real bitstreams, validation; seed 0, base network): (bpp, PSNR dB)
REF = {"merged_l0.02_s1": (0.972, 36.45), "split_l0.02_s1": (1.859, 37.82),
       "wide_merged_l0.08": (1.470, 36.70), "wide_merged_l1.28": (1.755, 37.02)}

checks = []
def check(name, ok, info=""):
    checks.append(bool(ok)); print(("PASS" if ok else "FAIL"), "-", name, info, flush=True)

# --- path setup ---
hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/protocol.json", recursive=True)]
assert len(hits) == 1, f"dataset root not found: {hits}"
ROOT = hits[0]
if importlib.util.find_spec("compressai") is None:
    print("compressai missing, installing...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "compressai==1.2.8"], capture_output=True, text=True)
    importlib.invalidate_caches()
check("compressai available", importlib.util.find_spec("compressai") is not None)
g = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
check("2 GPUs visible", g.stdout.count("GPU ") == 2)
check("NO test folder", not os.path.exists(f"{ROOT}/dataset/test"))
t0 = time.time()
subprocess.run(f"find {ROOT}/dataset -name '*.npy' -print0 | xargs -0 -P 4 -n 20 cat > /dev/null", shell=True)
print(f"cache warm-up took {time.time()-t0:.0f} s", flush=True)

WORKER = f"{OUT}/worker.py"
open(WORKER, "w").write(r'''
import os, sys, glob, time, json, random, gc
gpu, tag, mode, lam, steps, every, n_eval, OUT = (sys.argv[1], sys.argv[2], sys.argv[3], float(sys.argv[4]),
    int(sys.argv[5]), int(sys.argv[6]), int(sys.argv[7]), sys.argv[8])
N, M, seed = int(sys.argv[9]), int(sys.argv[10]), int(sys.argv[11])
os.environ["CUDA_VISIBLE_DEVICES"] = gpu
hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/protocol.json", recursive=True)]
assert len(hits) == 1, hits
ROOT = hits[0]
os.environ["DIPCOMA_ROOT"] = ROOT
sys.dont_write_bytecode = True
sys.path.insert(0, ROOT + "/code")
import numpy as np, pandas as pd, torch
import dipcoma_eval as E
import dipcoma_learned as L
L.DST = E.DATA
if os.path.exists(f"{OUT}/summary_{tag}.json"):
    print(f"{tag}: already done, skipping", flush=True); sys.exit(0)
train_p, val_p = E.list_patches("train"), E.list_patches("val")
ch = 13 if mode == "merged" else 1

@torch.no_grad()
def validate(model):
    model.eval()
    vl = L.make_loader(val_p, mode, False, 16, workers=0)
    bits, sq, cnt = 0.0, 0.0, 0
    for x in vl:
        x = x.to("cuda")
        out = model(x)
        bits += sum(-torch.log2(l).sum().item() for l in out["likelihoods"].values())
        sq += ((out["x_hat"].clamp(0, 1) - x) ** 2).sum().item()
        cnt += x.numel()
    return bits / cnt, float(10 * np.log10(1.0 / (sq / cnt)))

random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
loader = L.make_loader(train_p, mode, True, 16, workers=2)
batches = L.infinite(loader)
model = L.make_model(ch, N=N, M=M).to("cuda")
nparams = sum(p.numel() for p in model.parameters())
print(f"{tag}: N={N} M={M} seed={seed} params {nparams:,}", flush=True)
opt, aux_opt = L.make_optimizers(model)
ckpt, curvef = f"{OUT}/ckpt_{tag}.pt", f"{OUT}/curve_{tag}.json"
step, rows = 0, []
if os.path.exists(ckpt) and os.path.exists(curvef):
    ck = L.load_ckpt(ckpt, model, opt, aux_opt)
    step, rows = ck["step"], json.load(open(curvef))["rows"]
    print(f"{tag}: resumed at step {step}", flush=True)
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
while step < steps:
    n = min(every, steps - step)
    hist, _ = L.train_steps(model, opt, aux_opt, batches, n, lam, "cuda", log_every=n)
    step += n
    vb, vp = validate(model)
    rows.append(dict(step=step, val_bps=vb, val_psnr=vp, train_psnr=float(10 * np.log10(1 / hist[:, 2].mean())),
                     finite=bool(np.isfinite(hist).all()), minutes=(time.time() - t0) / 60))
    json.dump(dict(tag=tag, mode=mode, lam=lam, rows=rows), open(curvef, "w"))
    L.save_ckpt(ckpt, model, opt, aux_opt, step, mode, lam)       # saved BEFORE model.update()
    print(f"{tag} step {step}: val bits/sample {vb:.4f} | val PSNR {vp:.2f} dB | {rows[-1]['minutes']:.0f} min", flush=True)
peak = torch.cuda.max_memory_allocated() / 1e9
del loader, batches; gc.collect()
re = L.evaluate_real(model, mode, val_p, "cuda", n=n_eval)       # real bitstreams
df = pd.DataFrame(re); df.to_csv(f"{OUT}/eval_{tag}.csv", index=False)
summ = dict(tag=tag, mode=mode, lam=lam, steps=steps, N=N, M=M, seed=seed, params=nparams, peak_gb=peak,
            val_bps_est=rows[-1]["val_bps"], val_psnr_est=rows[-1]["val_psnr"],
            finite=all(r["finite"] for r in rows), n_eval=len(df),
            real_bps=float(df.bits_per_sample.mean()), real_bpp=float(df.bpp.mean()),
            CR=float(E.compression_ratio(df.bytes.mean())), psnr=float(df.psnr.mean()), ssim=float(df.ssim.mean()),
            est_min=float(df.est_over_actual.min()), est_max=float(df.est_over_actual.max()),
            t_enc=float(df.t_enc.median()), t_dec=float(df.t_dec.median()), minutes=(time.time() - t0) / 60)
json.dump(summ, open(f"{OUT}/summary_{tag}.json", "w"))
print(f"{tag}: DONE", flush=True)
''')

def last_line(tag):
    p = f"{OUT}/log_{tag}.txt"
    if not os.path.exists(p): return f"{tag}: no log yet"
    ls = [l for l in open(p).read().splitlines() if "val bits" in l or "Error" in l or "Traceback" in l]
    return ls[-1] if ls else f"{tag}: no validation line yet"

if all(checks):
    T0, queues, running, codes, lastp = time.time(), {k: list(v) for k, v in JOBS.items()}, {}, {}, 0.0
    while any(queues.values()) or running:
        for gi in (0, 1):
            if gi not in running and queues[gi]:
                tag, mode, lam, steps, N, M, seed = queues[gi].pop(0)
                log = open(f"{OUT}/log_{tag}.txt", "a")
                p = subprocess.Popen([sys.executable, WORKER, str(gi), tag, mode, str(lam), str(steps), str(EVERY),
                                      str(N_EVAL), OUT, str(N), str(M), str(seed)],
                                     stdout=log, stderr=subprocess.STDOUT, env=dict(os.environ, PYTHONUNBUFFERED="1"))
                running[gi] = (p, tag)
                print(f"[{(time.time()-T0)/60:.0f} min] started {tag} on GPU {gi}", flush=True)
        time.sleep(30)
        for gi, (p, tag) in list(running.items()):
            if p.poll() is not None:
                codes[tag] = p.returncode
                print(f"[{(time.time()-T0)/60:.0f} min] finished {tag}, exit code {p.returncode}", flush=True)
                del running[gi]
        if time.time() - lastp > 600:
            lastp = time.time()
            print(f"--- {(time.time()-T0)/60:.0f} min elapsed ---", flush=True)
            for gi, (p, tag) in running.items(): print("  ", last_line(tag), flush=True)

    tags = [j[0] for q in JOBS.values() for j in q]
    S = {t: json.load(open(f"{OUT}/summary_{t}.json")) for t in tags if os.path.exists(f"{OUT}/summary_{t}.json")}
    check("all jobs exited with code 0", len(codes) == len(tags) and all(c == 0 for c in codes.values()), str(codes))
    check("all summaries written", len(S) == len(tags), f"{len(S)}/{len(tags)}")
    if S:
        print(f"\n{'run':>18} {'lambda':>7} {'seed':>4} {'N,M':>8} {'params':>10} | est(val crops) bits/sample, dB | real(full patches): bits/sample, bpp, CR, PSNR, SSIM | minutes")
        for t, s in S.items():
            print(f"{t:>18} {s['lam']:>7g} {s['seed']:>4} {str(s['N'])+','+str(s['M']):>8} {s['params']:>10,} | {s['val_bps_est']:.4f}, {s['val_psnr_est']:.2f} | "
                  f"{s['real_bps']:.4f}, {s['real_bpp']:.3f}, {s['CR']:.0f}, {s['psnr']:.2f}, {s['ssim']:.3f} | {s['minutes']:.0f}")
        for t, (rb, rp) in REF.items():
            if t in S and not TEST:
                print(f"   vs earlier run for {t}: bpp {S[t]['real_bpp']:.3f} vs {rb:.3f} ({S[t]['real_bpp']-rb:+.3f}), "
                      f"PSNR {S[t]['psnr']:.2f} vs {rp:.2f} ({S[t]['psnr']-rp:+.2f} dB)")
        print("training speed lines (s/step, average over each block of steps):")
        for t in tags:
            p = f"{OUT}/log_{t}.txt"
            if os.path.exists(p):
                print(f"  {t}:", *[l.split('|')[-1].strip() for l in open(p).read().splitlines() if "s/step" in l][:4])
        check("all finite", all(s["finite"] for s in S.values()))
        check("real bytes match estimate within 10%", all(s["est_min"] > 0.9 and s["est_max"] < 1.1 for s in S.values()))
        print(f"INFO - total GPU minutes {sum(s['minutes'] for s in S.values()):.0f}, wall-clock {(time.time()-T0)/60:.0f} min")

print("\nALL PASS" if all(checks) else "\nSOME CHECKS FAILED - paste the whole output here")

compressai missing, installing...
PASS - compressai available 
PASS - 2 GPUs visible 
PASS - NO test folder 
cache warm-up took 25 s
[0 min] started wide_merged_l0.08 on GPU 0
[0 min] started wide_merged_l1.28 on GPU 1
--- 1 min elapsed ---
   wide_merged_l0.08: no validation line yet
   wide_merged_l1.28: no validation line yet
--- 11 min elapsed ---
   wide_merged_l0.08: no validation line yet
   wide_merged_l1.28 step 1000: val bits/sample 0.0704 | val PSNR 28.00 dB | 10 min
--- 21 min elapsed ---
   wide_merged_l0.08 step 1000: val bits/sample 0.0551 | val PSNR 27.45 dB | 11 min
   wide_merged_l1.28 step 2000: val bits/sample 0.0828 | val PSNR 30.17 dB | 20 min
--- 31 min elapsed ---
   wide_merged_l0.08 step 2000: val bits/sample 0.0732 | val PSNR 29.94 dB | 22 min
   wide_merged_l1.28 step 3000: val bits/sample 0.0997 | val PSNR 31.84 dB | 30 min
--- 41 min elapsed ---
   wide_merged_l0.08 step 3000: val bits/sample 0.0828 | val PSNR 31.68 dB | 33 min
   wide_merged_l1.28 step 40